In [1]:

# ═══════════════════════════════════════════════════════════════
# RESTART RECOVERY — Run this single cell after restart!
# Reinstalls + restores everything automatically
# ═══════════════════════════════════════════════════════════════

import subprocess, os, zipfile, time, json

# ─── Step 1: Install packages ───
print("=" * 55)
print("  STEP 1: Installing packages...")
print("=" * 55)

subprocess.run([
    "pip", "install", "-q",
    "torchao>=0.16.0",
    "colpali-engine",
    "chromadb",
    "sentence-transformers",
    "accelerate",
    "bitsandbytes",
    "transformers",
    "Pillow",
    "pymupdf",
], check=False)

subprocess.run(
    ["apt-get", "install", "-y", "-q", "poppler-utils"],
    check=False
)
print("  ✅ Packages installed!")

# ─── Step 2: Restore files ───
print("\n" + "=" * 55)
print("  STEP 2: Restoring files...")
print("=" * 55)

dirs = [
    "data/raw", "data/parsed/pages",
    "data/parsed/markdown",
    "data/indices/multivectors",
    "data/indices/chroma_index",
    "outputs",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

idx_zip   = "/content/sci-rag-indices.zip"
pages_zip = "/content/sci-rag-pages.zip"

if os.path.exists(idx_zip):
    with zipfile.ZipFile(idx_zip, "r") as zf:
        zf.extractall("data")
    print(f"  ✅ Indices restored!")
else:
    print("  ❌ sci-rag-indices.zip missing!")

if os.path.exists(pages_zip):
    with zipfile.ZipFile(pages_zip, "r") as zf:
        zf.extractall("data")
    print(f"  ✅ Pages restored!")
else:
    print("  ❌ sci-rag-pages.zip missing!")

# ─── Step 3: Verify packages ───
print("\n" + "=" * 55)
print("  STEP 3: Verifying...")
print("=" * 55)

import torch
import numpy as np
import torchao

print(f"  torch          : {torch.__version__}")
print(f"  torchao        : {torchao.__version__}")

if torch.cuda.is_available():
    print(f"  GPU            : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"  VRAM total     : {vram:.1f} GB")
    print(f"  VRAM free      : {free:.1f} GB")

try:
    from colpali_engine.models import ColPali, ColPaliProcessor
    print("  colpali-engine : ✅")
except Exception as e:
    print(f"  colpali-engine : ❌ {e}")

try:
    from sentence_transformers import SentenceTransformer
    print("  sentence-trans : ✅")
except Exception as e:
    print(f"  sentence-trans : ❌ {e}")

try:
    import chromadb
    print(f"  chromadb       : {chromadb.__version__} ✅")
except Exception as e:
    print(f"  chromadb       : ❌ {e}")

# ─── Step 4: Load metadata + ChromaDB ───
print("\n" + "=" * 55)
print("  STEP 4: Loading metadata...")
print("=" * 55)

import chromadb

CONFIG = {
    "npy_dir"             : "data/indices/multivectors",
    "chroma_dir"          : "data/indices/chroma_index",
    "pages_dir"           : "data/parsed/pages",
    "colpali_model"       : "vidore/colpali-v1.2",
    "scincl_model"        : "malteos/scincl",
    "qwen_model"          : "Qwen/Qwen2-VL-2B-Instruct",
    "top_k"               : 3,
    "colpali_weight"      : 0.7,
    "scincl_weight"       : 0.3,
    "confidence_threshold": 0.5,
    "max_retries"         : 1,
}

with open("data/indices/page_metadata.json") as f:
    page_metadata = json.load(f)
with open("data/indices/doc_mapping.json") as f:
    doc_mapping = json.load(f)
with open("data/indices/summary.json") as f:
    summary = json.load(f)

print(f"  ✅ page_metadata : {len(page_metadata)} pages")
print(f"  ✅ doc_mapping   : {len(doc_mapping)} papers")

chroma_client = chromadb.PersistentClient(path=CONFIG["chroma_dir"])
collections   = chroma_client.list_collections()
collection    = chroma_client.get_collection(collections[0].name)
print(f"  ✅ ChromaDB      : {collection.count()} entries")

npy_index = {}
for fname in os.listdir(CONFIG["npy_dir"]):
    if fname.endswith(".npy") and not fname.endswith(".meta.npy"):
        page_key = fname.replace(".npy", "")
        npy_index[page_key] = os.path.join(CONFIG["npy_dir"], fname)
print(f"  ✅ npy_index     : {len(npy_index)} files")

print(f"\n{'='*55}")
print(f"  ✅ ALL READY!")
print(f"  GPU  : {torch.cuda.get_device_name(0)}")
print(f"  VRAM : {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB free")
print(f"{'='*55}")
print("✅ RECOVERY CELL COMPLETE — Now run Cell 4 + 5 + 6!")

  STEP 1: Installing packages...
  ✅ Packages installed!

  STEP 2: Restoring files...
  ✅ Indices restored!
  ✅ Pages restored!

  STEP 3: Verifying...


  torch          : 2.10.0+cu128
  torchao        : 0.17.0
  GPU            : Tesla T4
  VRAM total     : 14.6 GB
  VRAM free      : 14.5 GB
  colpali-engine : ✅
  sentence-trans : ✅
  chromadb       : 1.5.9 ✅

  STEP 4: Loading metadata...
  ✅ page_metadata : 182 pages
  ✅ doc_mapping   : 10 papers
  ✅ ChromaDB      : 182 entries
  ✅ npy_index     : 182 files

  ✅ ALL READY!
  GPU  : Tesla T4
  VRAM : 14.5 GB free
✅ RECOVERY CELL COMPLETE — Now run Cell 4 + 5 + 6!


In [2]:

# ═══════════════════════════════════════════════════════════════
# CELL 4: Retrieval Functions
# ColPali Visual + SciNCL Text + Score Fusion
# ═══════════════════════════════════════════════════════════════

import gc
import numpy as np
import torch
from PIL import Image
from colpali_engine.models import ColPali, ColPaliProcessor
from sentence_transformers import SentenceTransformer

print("=" * 55)
print("  CELL 4: RETRIEVAL FUNCTIONS")
print("=" * 55)

# ═══════════════════════════════════════
# FUNCTION 1: ColPali Visual Retrieval
# ═══════════════════════════════════════

def colpali_retrieve(query, top_k=5):
    """
    Load ColPali → encode query → MaxSim score
    all pages → unload → return top_k results
    """
    print(f"  [ColPali] Loading model...")
    t0 = torch.cuda.mem_get_info()[0] / 1024**3

    # Load
    model = ColPali.from_pretrained(
        CONFIG["colpali_model"],
        torch_dtype=torch.float16,
        device_map="cuda",
        low_cpu_mem_usage=True,
    )
    processor = ColPaliProcessor.from_pretrained(
        CONFIG["colpali_model"]
    )
    model.eval()

    t1 = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"  [ColPali] Loaded! VRAM used: {t0-t1:.1f} GB")

    # Encode query
    batch = processor.process_queries(queries=[query])
    batch = {k: v.to(model.device) for k, v in batch.items()}

    with torch.no_grad():
        query_emb = model(**batch)

    query_vec = query_emb[0].cpu().float().numpy()
    del batch, query_emb
    torch.cuda.empty_cache()

    # Score all pages via MaxSim
    print(f"  [ColPali] Scoring {len(npy_index)} pages...")
    scores = []

    for page_key, npy_path in npy_index.items():
        try:
            page_vec   = np.load(npy_path)
            sim_matrix = np.matmul(query_vec, page_vec.T)
            score      = float(sim_matrix.max(axis=1).mean())
            scores.append((page_key, score))
        except Exception as e:
            continue

    scores.sort(key=lambda x: x[1], reverse=True)
    top_scores = scores[:top_k]

    # Unload
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  [ColPali] Done! Top score: {top_scores[0][1]:.4f}")

    # Build results
    results = []
    for page_key, score in top_scores:
        meta = page_metadata.get(page_key, {})
        results.append({
            "page_key"   : page_key,
            "score"      : score,
            "doc_id"     : meta.get("doc_id", ""),
            "page_num"   : meta.get("page_num", 0),
            "paper_title": meta.get("paper_title", ""),
            "image_path" : meta.get("image_path", ""),
            "text"       : meta.get("text", ""),
        })

    return results


# ═══════════════════════════════════════
# FUNCTION 2: SciNCL Text Retrieval
# ═══════════════════════════════════════

def scincl_retrieve(query, top_k=5):
    """
    Load SciNCL → encode query → ChromaDB search
    → unload → return top_k results
    """
    print(f"  [SciNCL] Loading model...")

    # Load
    model = SentenceTransformer(
        CONFIG["scincl_model"],
        device="cuda"
    )

    # Encode
    query_vec = model.encode(
        query,
        convert_to_numpy=True
    )

    # Search ChromaDB
    results_db = collection.query(
        query_embeddings=[query_vec.tolist()],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    # Unload
    del model
    gc.collect()
    torch.cuda.empty_cache()

    # Build results
    results = []
    for i, doc_id in enumerate(results_db["ids"][0]):
        meta  = results_db["metadatas"][0][i]
        dist  = results_db["distances"][0][i]
        sim   = 1 - dist
        page_meta = page_metadata.get(doc_id, {})
        results.append({
            "page_key"   : doc_id,
            "score"      : sim,
            "doc_id"     : meta.get("doc_id", ""),
            "page_num"   : int(meta.get("page_num", 0)),
            "paper_title": meta.get("paper_title", ""),
            "image_path" : page_meta.get("image_path", ""),
            "text"       : page_meta.get("text", ""),
        })

    print(f"  [SciNCL] Done! Top score: {results[0]['score']:.4f}")
    return results


# ═══════════════════════════════════════
# FUNCTION 3: Score Fusion
# ═══════════════════════════════════════

def fuse_scores(colpali_results, scincl_results,
                colpali_weight=0.7, scincl_weight=0.3,
                top_k=3):
    """
    Normalize + weight + fuse ColPali and SciNCL scores
    Returns top_k fused results
    """

    # Normalize ColPali
    c_scores = {r["page_key"]: r["score"] for r in colpali_results}
    c_vals   = list(c_scores.values())
    c_min, c_max = min(c_vals), max(c_vals)

    # Normalize SciNCL
    s_scores = {r["page_key"]: r["score"] for r in scincl_results}
    s_vals   = list(s_scores.values())
    s_min, s_max = min(s_vals), max(s_vals)

    # Fuse
    fused = {}
    for pk, sc in c_scores.items():
        norm_c    = (sc - c_min) / (c_max - c_min + 1e-8)
        fused[pk] = colpali_weight * norm_c

    for pk, sc in s_scores.items():
        norm_s = (sc - s_min) / (s_max - s_min + 1e-8)
        if pk in fused:
            fused[pk] += scincl_weight * norm_s
        else:
            fused[pk]  = scincl_weight * norm_s

    # Sort
    fused_sorted = sorted(
        fused.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    # Build final results
    all_results = {r["page_key"]: r for r in colpali_results}
    all_results.update({r["page_key"]: r for r in scincl_results})

    final = []
    for page_key, fused_score in fused_sorted:
        r = all_results.get(page_key, {})
        r["fused_score"] = fused_score
        final.append(r)

    return final


# ═══════════════════════════════════════
# FUNCTION 4: Full Retrieve Pipeline
# ═══════════════════════════════════════

def retrieve(query, top_k=3):
    """
    Full retrieval pipeline:
    ColPali + SciNCL + Fusion → top_k pages
    """
    print(f"\n  Query: '{query[:60]}'")
    print(f"  {'─'*50}")

    t_start = time.time()

    # Step 1: ColPali
    colpali_results = colpali_retrieve(query, top_k=10)

    # Step 2: SciNCL
    scincl_results  = scincl_retrieve(query, top_k=10)

    # Step 3: Fusion
    fused_results   = fuse_scores(
        colpali_results,
        scincl_results,
        colpali_weight=CONFIG["colpali_weight"],
        scincl_weight =CONFIG["scincl_weight"],
        top_k=top_k,
    )

    t_end = time.time()

    print(f"\n  [Fusion] Top {top_k} results:")
    for i, r in enumerate(fused_results):
        print(f"    [{i+1}] {r['page_key']}")
        print(f"         {r['paper_title'][:50]}")
        print(f"         fused={r['fused_score']:.4f}")

    print(f"\n  Retrieval time: {t_end-t_start:.1f}s")
    return fused_results

print(f"\n{'='*55}")
print("  ✅ FUNCTIONS LOADED!")
print("  ├── colpali_retrieve(query, top_k)")
print("  ├── scincl_retrieve(query, top_k)")
print("  ├── fuse_scores(c_results, s_results)")
print("  └── retrieve(query, top_k)  ← main function")
print(f"{'='*55}")
print("✅ CELL 4 COMPLETE!")

  CELL 4: RETRIEVAL FUNCTIONS

  ✅ FUNCTIONS LOADED!
  ├── colpali_retrieve(query, top_k)
  ├── scincl_retrieve(query, top_k)
  ├── fuse_scores(c_results, s_results)
  └── retrieve(query, top_k)  ← main function
✅ CELL 4 COMPLETE!


In [3]:

# ═══════════════════════════════════════════════════════════════
# CELL 5: Qwen2-VL Answer Generation Function
# ═══════════════════════════════════════════════════════════════

import torch
import gc
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

print("=" * 55)
print("  CELL 5: QWEN2-VL GENERATION FUNCTION")
print("=" * 55)

# ═══════════════════════════════════════
# FUNCTION: Generate Answer with Qwen2-VL
# ═══════════════════════════════════════

def generate_answer(query, retrieved_pages):
    """
    Load Qwen2-VL → build prompt with images + text
    → generate answer → unload → return answer
    """

    print(f"  [Qwen2-VL] Loading model...")
    t0 = torch.cuda.mem_get_info()[0] / 1024**3

    # ─── Load Qwen2-VL ───
    qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
        CONFIG["qwen_model"],
        torch_dtype=torch.float16,
        device_map="cuda",
        low_cpu_mem_usage=True,
    )
    qwen_processor = AutoProcessor.from_pretrained(
        CONFIG["qwen_model"]
    )
    qwen_model.eval()

    t1 = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"  [Qwen2-VL] Loaded! VRAM used: {t0-t1:.1f} GB")

    # ─── Build context from retrieved pages ───
    context_texts = []
    context_images = []

    for i, page in enumerate(retrieved_pages[:3]):
        # Add text context
        text = page.get("text", "").strip()[:400]
        title = page.get("paper_title", "")
        page_num = page.get("page_num", 0)

        context_texts.append(
            f"[Source {i+1}] {title} — Page {page_num}:\n{text}"
        )

        # Add image if exists
        img_path = page.get("image_path", "")
        if img_path and os.path.exists(img_path):
            try:
                img = Image.open(img_path).convert("RGB")
                # Resize to save VRAM
                img = img.resize((512, 512), Image.LANCZOS)
                context_images.append(img)
            except:
                pass

    context_str = "\n\n".join(context_texts)

    print(f"  [Qwen2-VL] Context: {len(context_images)} images, "
          f"{len(context_texts)} text chunks")

    # ─── Build messages ───
    # Build content list with images first then text
    content = []

    for img in context_images:
        content.append({
            "type" : "image",
            "image": img,
        })

    content.append({
        "type": "text",
        "text": f"""You are a scientific paper assistant.
Answer the question based on the provided paper pages.
Be specific and cite which paper/page supports your answer.

Context from papers:
{context_str}

Question: {query}

Answer:"""
    })

    messages = [
        {
            "role"   : "user",
            "content": content,
        }
    ]

    # ─── Process + Generate ───
    try:
        text_input = qwen_processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        # Get images list for processor
        images_for_proc = context_images if context_images else None

        inputs = qwen_processor(
            text=text_input,
            images=images_for_proc,
            return_tensors="pt",
        ).to(qwen_model.device)

        print(f"  [Qwen2-VL] Generating answer...")

        with torch.no_grad():
            output_ids = qwen_model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=False,
                temperature=1.0,
                repetition_penalty=1.1,
            )

        # Decode only new tokens
        input_len = inputs["input_ids"].shape[1]
        new_tokens = output_ids[0][input_len:]
        answer = qwen_processor.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

    except Exception as e:
        print(f"  [Qwen2-VL] ⚠️ Error: {e}")
        answer = f"Error generating answer: {e}"

    finally:
        # ─── Unload Qwen2-VL ───
        print(f"  [Qwen2-VL] Unloading...")
        del qwen_model, qwen_processor
        gc.collect()
        torch.cuda.empty_cache()
        freed = torch.cuda.mem_get_info()[0] / 1024**3
        print(f"  [Qwen2-VL] Unloaded! VRAM free: {freed:.1f} GB")

    return answer


# ═══════════════════════════════════════
# FULL RAG PIPELINE FUNCTION
# ═══════════════════════════════════════

def rag_query(query):
    """
    Complete RAG Pipeline:
    Query → Retrieve → Generate → Return
    """
    print(f"\n{'='*55}")
    print(f"  RAG QUERY")
    print(f"{'='*55}")
    print(f"  Q: {query}")
    print(f"{'='*55}")

    t_start = time.time()

    # ─── Step 1: Retrieve ───
    print(f"\n  STEP 1: RETRIEVAL")
    retrieved = retrieve(query, top_k=CONFIG["top_k"])

    # ─── Step 2: Generate ───
    print(f"\n  STEP 2: GENERATION")
    answer = generate_answer(query, retrieved)

    t_total = time.time() - t_start

    # ─── Step 3: Build output ───
    sources = []
    for r in retrieved:
        sources.append({
            "paper"      : r.get("paper_title", "")[:50],
            "page"       : r.get("page_num", 0),
            "arxiv_id"   : r.get("doc_id", ""),
            "fused_score": round(r.get("fused_score", 0), 4),
        })

    result = {
        "query"      : query,
        "answer"     : answer,
        "sources"    : sources,
        "total_time" : round(t_total, 1),
    }

    # ─── Print result ───
    print(f"\n{'='*55}")
    print(f"  ✅ ANSWER ({t_total:.1f}s)")
    print(f"{'='*55}")
    print(f"\n  {answer}")
    print(f"\n  📚 Sources:")
    for s in sources:
        print(f"    - {s['paper']}")
        print(f"      Page {s['page']} | "
              f"arxiv:{s['arxiv_id']} | "
              f"score:{s['fused_score']}")
    print(f"{'='*55}")

    return result

print(f"\n{'='*55}")
print("  ✅ GENERATION FUNCTIONS LOADED!")
print("  ├── generate_answer(query, pages)")
print("  └── rag_query(query)  ← USE THIS!")
print(f"{'='*55}")
print("✅ CELL 5 COMPLETE!")

  CELL 5: QWEN2-VL GENERATION FUNCTION

  ✅ GENERATION FUNCTIONS LOADED!
  ├── generate_answer(query, pages)
  └── rag_query(query)  ← USE THIS!
✅ CELL 5 COMPLETE!


In [4]:

# ═══════════════════════════════════════════════════════════════
# CELL 6: FIRST REAL RAG TEST QUERY
# Full pipeline: Retrieve + Generate Answer
# Time: ~1-2 minutes
# ═══════════════════════════════════════════════════════════════

print("=" * 55)
print("  CELL 6: FIRST RAG TEST")
print("=" * 55)

# ─── Run first query ───
result = rag_query("What is the Vision Transformer and how does it work?")

# ─── Pretty print ───
print(f"\n{'═'*55}")
print(f"  FINAL RESULT SUMMARY")
print(f"{'═'*55}")
print(f"  Query      : {result['query']}")
print(f"  Total time : {result['total_time']}s")
print(f"\n  Answer:")
print(f"  {result['answer']}")
print(f"\n  Sources used:")
for i, s in enumerate(result['sources']):
    print(f"  [{i+1}] {s['paper']}")
    print(f"       Page {s['page']} | "
          f"https://arxiv.org/abs/{s['arxiv_id']} | "
          f"score: {s['fused_score']}")
print(f"{'═'*55}")
print("✅ CELL 6 COMPLETE!")

  CELL 6: FIRST RAG TEST

  RAG QUERY
  Q: What is the Vision Transformer and how does it work?

  STEP 1: RETRIEVAL

  Query: 'What is the Vision Transformer and how does it work?'
  ──────────────────────────────────────────────────
  [ColPali] Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/605 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/78.6M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  [Qwen2-VL] Loaded! VRAM used: 0.1 GB
  [Qwen2-VL] Context: 3 images, 3 text chunks
  [Qwen2-VL] Generating answer...
  [Qwen2-VL] Unloading...
  [Qwen2-VL] Unloaded! VRAM free: 9.0 GB

  ✅ ANSWER (198.2s)

  The Vision Transformer (ViT) is a type of neural network used for vision tasks such as image classification, object detection, and semantic image segmentation. It works by taking an input image and using a series of layers to learn features from the image. The output of each layer is then fed into the next layer, allowing the network to learn more complex representations of the image.

  📚 Sources:
    - DeepViT: Towards Deeper Vision Transformer
      Page 2 | arxiv:2108.00102 | score:0.7
    - Training data-efficient image transformers & disti
      Page 1 | arxiv:2106.10270 | score:0.6181
    - EfficientFormer: Vision Transformers at MobileNet 
      Page 15 | arxiv:2203.14465 | score:0.4183

═══════════════════════════════════════════════════════
  FINAL RESULT SUMMARY
══════

In [5]:

# ═══════════════════════════════════════════════════════════════
# CELL 7: Improved Prompt + More Test Queries
# Better answers with detailed prompting
# ═══════════════════════════════════════════════════════════════

import gc
import os
import torch
import numpy as np
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

print("=" * 55)
print("  CELL 7: IMPROVED GENERATION + MORE TESTS")
print("=" * 55)

# ─── Improved generate function ───
def generate_answer_v2(query, retrieved_pages):
    """
    Improved Qwen2-VL with better prompt
    """
    print(f"  [Qwen2-VL] Loading model...")

    qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
        CONFIG["qwen_model"],
        torch_dtype=torch.float16,
        device_map="cuda",
        low_cpu_mem_usage=True,
    )
    qwen_processor = AutoProcessor.from_pretrained(
        CONFIG["qwen_model"]
    )
    qwen_model.eval()

    free = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"  [Qwen2-VL] Loaded! VRAM free: {free:.1f} GB")

    # ─── Build context ───
    context_texts  = []
    context_images = []

    for i, page in enumerate(retrieved_pages[:3]):
        text     = page.get("text", "").strip()[:600]
        title    = page.get("paper_title", "Unknown")
        page_num = page.get("page_num", 0)
        doc_id   = page.get("doc_id", "")

        context_texts.append(
            f"[Paper {i+1}: {title} | arXiv:{doc_id} | Page {page_num}]\n"
            f"{text}"
        )

        img_path = page.get("image_path", "")
        if img_path and os.path.exists(img_path):
            try:
                img = Image.open(img_path).convert("RGB")
                img = img.resize((448, 448), Image.LANCZOS)
                context_images.append(img)
            except:
                pass

    context_str = "\n\n---\n\n".join(context_texts)

    print(f"  [Qwen2-VL] Context: "
          f"{len(context_images)} images, "
          f"{len(context_texts)} chunks")

    # ─── Improved prompt ───
    prompt = f"""You are an expert scientific paper analyst specializing in Vision Transformers and deep learning.

You have been given {len(context_images)} page images and text from relevant research papers.

CONTEXT FROM PAPERS:
{context_str}

QUESTION: {query}

Instructions:
- Answer specifically based on the paper content above
- Mention specific details like architecture names, numbers, datasets
- Cite which paper supports each claim (e.g. "According to ViT paper...")
- Be detailed and technical but clear
- If the papers don't contain enough info, say so honestly

ANSWER:"""

    # ─── Build messages ───
    content = []
    for img in context_images:
        content.append({"type": "image", "image": img})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]

    # ─── Generate ───
    try:
        text_input = qwen_processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = qwen_processor(
            text=text_input,
            images=context_images if context_images else None,
            return_tensors="pt",
        ).to(qwen_model.device)

        print(f"  [Qwen2-VL] Generating...")

        with torch.no_grad():
            output_ids = qwen_model.generate(
                **inputs,
                max_new_tokens=400,
                do_sample=False,
                repetition_penalty=1.1,
            )

        input_len  = inputs["input_ids"].shape[1]
        new_tokens = output_ids[0][input_len:]
        answer     = qwen_processor.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

    except Exception as e:
        print(f"  ❌ Error: {e}")
        answer = f"Error: {e}"

    finally:
        del qwen_model, qwen_processor
        gc.collect()
        torch.cuda.empty_cache()
        free = torch.cuda.mem_get_info()[0] / 1024**3
        print(f"  [Qwen2-VL] Unloaded! VRAM free: {free:.1f} GB")

    return answer


# ─── Improved RAG pipeline ───
def rag_query_v2(query):
    """
    Full RAG with improved generation
    """
    print(f"\n{'═'*55}")
    print(f"  RAG QUERY v2")
    print(f"{'═'*55}")
    print(f"  Q: {query}")
    print(f"{'═'*55}")

    import time
    t_start = time.time()

    # Retrieve
    print(f"\n  STEP 1: RETRIEVAL")
    retrieved = retrieve(query, top_k=CONFIG["top_k"])

    # Generate
    print(f"\n  STEP 2: GENERATION")
    answer = generate_answer_v2(query, retrieved)

    t_total = time.time() - t_start

    # Sources
    sources = []
    for r in retrieved:
        sources.append({
            "paper"      : r.get("paper_title", "")[:55],
            "page"       : r.get("page_num", 0),
            "arxiv_id"   : r.get("doc_id", ""),
            "fused_score": round(r.get("fused_score", 0), 4),
        })

    # Print result
    print(f"\n{'═'*55}")
    print(f"  ✅ ANSWER ({t_total:.1f}s)")
    print(f"{'═'*55}")
    print(f"\n{answer}")
    print(f"\n  📚 Sources:")
    for i, s in enumerate(sources):
        print(f"  [{i+1}] {s['paper']}")
        print(f"       Page {s['page']} | "
              f"arxiv:{s['arxiv_id']} | "
              f"score:{s['fused_score']}")
    print(f"{'═'*55}")

    return {
        "query"     : query,
        "answer"    : answer,
        "sources"   : sources,
        "total_time": round(t_total, 1),
    }


print("✅ CELL 7 LOADED — Running test queries...\n")

# ─── Test Query 1 ───
r1 = rag_query_v2(
    "How does patch embedding work in Vision Transformer?"
)

# ─── Test Query 2 ───
r2 = rag_query_v2(
    "What is the difference between ViT and CNN?"
)

print("\n✅ CELL 7 COMPLETE!")

  CELL 7: IMPROVED GENERATION + MORE TESTS
✅ CELL 7 LOADED — Running test queries...


═══════════════════════════════════════════════════════
  RAG QUERY v2
═══════════════════════════════════════════════════════
  Q: How does patch embedding work in Vision Transformer?
═══════════════════════════════════════════════════════

  STEP 1: RETRIEVAL

  Query: 'How does patch embedding work in Vision Transformer?'
  ──────────────────────────────────────────────────
  [ColPali] Loading model...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/605 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] ColPali LOAD REPORT from: vidore/colpali-v1.2
Key                                                                               | Status     | 
----------------------------------------------------------------------------------+------------+-
model.language_model.model.layers.{0...17}.self_attn.v_proj.lora_B.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.q_proj.lora_A.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.k_proj.lora_A.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.o_proj.lora_B.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.v_proj.lora_A.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.mlp.up_proj.lora_A.default.weight      | UNEXPECTED | 
model.language_model.model.layers.{0...17}.mlp.gate_proj.lora_A.default.weight    | UNEXPECTED | 
model.language_model.model.layers.{0...17}.mlp.down_proj.

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [SciNCL] Done! Top score: 0.9182

  [Fusion] Top 3 results:
    [1] 2108.00102_page_2
         DeepViT: Towards Deeper Vision Transformer
         fused=0.7000
    [2] 2203.14465_page_15
         EfficientFormer: Vision Transformers at MobileNet 
         fused=0.3771
    [3] 2106.10270_page_1
         Training data-efficient image transformers & disti
         fused=0.3161

  Retrieval time: 39.5s

  STEP 2: GENERATION
  [Qwen2-VL] Loading model...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

  [Qwen2-VL] Loaded! VRAM free: 8.9 GB
  [Qwen2-VL] Context: 3 images, 3 chunks
  [Qwen2-VL] Generating...
  [Qwen2-VL] Unloaded! VRAM free: 9.0 GB

═══════════════════════════════════════════════════════
  ✅ ANSWER (80.0s)
═══════════════════════════════════════════════════════

Based on the information provided in the papers, the main differences between ViT and CNN can be summarized as follows:

1. **Architecture**: 
   - ViT uses a transformer architecture with multiple layers of self-attention blocks, while CNNs typically use convolutional layers without any form of self-attention or multi-layered structure.

2. **Number of Parameters**:
   - ViT has significantly fewer parameters compared to CNNs due to its compact architecture and efficient computation. For example, ViT models trained on ImageNet have around 10 times fewer parameters than CNNs trained on the same dataset.

3. **Training Efficiency**:
   - ViT models are faster to train compared to CNNs because they require less 